# Manual full taxi-data loader

Use this notebook **instead of `00_01_data_loader.ipynb`** when the Chicago taxi CSV has already been downloaded manually. It converts the localized manual export into the same bronze schema expected by `00_02_data_prep_silver.ipynb`.

The conversion is performed by DuckDB as a streaming CSV-to-Parquet operation, so the 7 GB file is not loaded completely into RAM. Other raw inputs and bronze files are left untouched.

In [1]:
from pathlib import Path

import duckdb

from run_config import PATHS, PROJECT_ROOT, RUN_MODE, START_DATE, END_DATE

if RUN_MODE != "full":
    raise ValueError("Set RUN_MODE = 'full' in run_config.py before running this notebook.")

MANUAL_CSV = PROJECT_ROOT / "data" / "Taxi_Trips_(2024-)_20260514.csv"
BRONZE_TAXI_PATH = PATHS.bronze_taxi_trips
EXPECTED_ROWS = 15_406_960
OVERWRITE = True

if not MANUAL_CSV.is_file():
    raise FileNotFoundError(f"Manual taxi CSV not found: {MANUAL_CSV}")

BRONZE_TAXI_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Input:  {MANUAL_CSV} ({MANUAL_CSV.stat().st_size / 1024**3:.2f} GiB)")
print(f"Output: {BRONZE_TAXI_PATH}")
print(f"Configured interval: [{START_DATE}, {END_DATE})")

Input:  /Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/Taxi_Trips_(2024-)_20260514.csv (6.65 GiB)
Output: /Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/processed_data/bronze_taxi_sample.parquet
Configured interval: [2024-01-01T00:00:00, 2026-05-01T00:00:00)


## Convert the manual export

The browser export uses display labels, German-style decimal commas, currency symbols, and no Socrata metadata fields. This query normalizes those differences. The four null metadata fields are intentional because the silver notebook expects and drops them.

In [2]:
if BRONZE_TAXI_PATH.exists() and not OVERWRITE:
    raise FileExistsError(
        f"Output already exists: {BRONZE_TAXI_PATH}. Set OVERWRITE=True to replace it."
    )

con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")
con.execute("SET threads = 4")

# Manual values look like '$46,25', '18,96', or '1.457'.
con.execute("""
CREATE OR REPLACE TEMP MACRO localized_number(value) AS
    TRY_CAST(
        REPLACE(REPLACE(REPLACE(TRIM(value), '$', ''), '.', ''), ',', '.')
        AS DOUBLE
    );
""")

query = f"""
COPY (
    SELECT
        "Trip ID" AS trip_id,
        "Taxi ID" AS taxi_id,
        TRY_STRPTIME("Trip Start Timestamp", '%m/%d/%Y %I:%M:%S %p') AS trip_start_timestamp,
        TRY_STRPTIME("Trip End Timestamp", '%m/%d/%Y %I:%M:%S %p') AS trip_end_timestamp,
        TRY_CAST(localized_number("Trip Seconds") AS BIGINT) AS trip_seconds,
        localized_number("Trip Miles") AS trip_miles,
        TRY_CAST(localized_number("Pickup Census Tract") AS BIGINT) AS pickup_census_tract,
        TRY_CAST(localized_number("Dropoff Census Tract") AS BIGINT) AS dropoff_census_tract,
        TRY_CAST(localized_number("Pickup Community Area") AS BIGINT) AS pickup_community_area,
        TRY_CAST(localized_number("Dropoff Community Area") AS BIGINT) AS dropoff_community_area,
        localized_number("Fare") AS fare,
        localized_number("Tips") AS tips,
        localized_number("Tolls") AS tolls,
        localized_number("Extras") AS extras,
        localized_number("Trip Total") AS trip_total,
        NULLIF(TRIM("Payment Type"), '') AS payment_type,
        NULLIF(TRIM("Company"), '') AS company,
        localized_number("Pickup Centroid Latitude") AS pickup_centroid_latitude,
        localized_number("Pickup Centroid Longitude") AS pickup_centroid_longitude,
        NULLIF(TRIM("Pickup Centroid Location"), '') AS pickup_centroid_location,
        localized_number("Dropoff Centroid Latitude") AS dropoff_centroid_latitude,
        localized_number("Dropoff Centroid Longitude") AS dropoff_centroid_longitude,
        NULLIF(TRIM("Dropoff Centroid  Location"), '') AS dropoff_centroid_location,
        NULL::VARCHAR AS ":id",
        NULL::VARCHAR AS ":version",
        NULL::TIMESTAMP AS ":created_at",
        NULL::TIMESTAMP AS ":updated_at"
    FROM read_csv(
        '{MANUAL_CSV.as_posix()}',
        header = true,
        all_varchar = true,
        strict_mode = true,
        null_padding = false
    )
) TO '{BRONZE_TAXI_PATH.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)
print("Conversion complete.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Conversion complete.


## Validate before continuing

This cell checks the row count, timestamp coverage, null timestamps, and output schema. It raises an error if the known manual file is not fully represented.

In [3]:
summary = con.execute(
    f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(trip_start_timestamp) AS min_timestamp,
        MAX(trip_start_timestamp) AS max_timestamp,
        COUNT(*) FILTER (WHERE trip_start_timestamp IS NULL) AS invalid_start_timestamps
    FROM read_parquet('{BRONZE_TAXI_PATH.as_posix()}')
    """
).fetchdf()
display(summary)

actual_rows = int(summary.loc[0, "row_count"])
invalid_timestamps = int(summary.loc[0, "invalid_start_timestamps"])
if actual_rows != EXPECTED_ROWS:
    raise RuntimeError(f"Expected {EXPECTED_ROWS:,} rows but wrote {actual_rows:,}.")
if invalid_timestamps:
    raise RuntimeError(f"Found {invalid_timestamps:,} invalid trip-start timestamps.")

display(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{BRONZE_TAXI_PATH.as_posix()}')").fetchdf())
print("Validation passed. You can now run 00_02_data_prep_silver.ipynb.")
con.close()

,row_count,min_timestamp,max_timestamp,invalid_start_timestamps
0,15406960,2024-01-01,2026-05-01,0


,column_name,column_type,null,key,default,extra
0,trip_id,VARCHAR,YES,None,None,None
1,taxi_id,VARCHAR,YES,None,None,None
2,trip_start_timestamp,TIMESTAMP,YES,None,None,None
3,trip_end_timestamp,TIMESTAMP,YES,None,None,None
4,trip_seconds,BIGINT,YES,None,None,None
5,trip_miles,DOUBLE,YES,None,None,None
6,pickup_census_tract,BIGINT,YES,None,None,None
7,dropoff_census_tract,BIGINT,YES,None,None,None
8,pickup_community_area,BIGINT,YES,None,None,None
9,dropoff_community_area,BIGINT,YES,None,None,None


Validation passed. You can now run 00_02_data_prep_silver.ipynb.
